# Feature Track 0b: Document Ingestion and Chunking

---

This notebook explores how raw documents are converted into chunks that the RAG pipeline can retrieve and reason over. It is a companion to `feature0a_baseline_rag.ipynb`, which covers the full pipeline end-to-end.

**What this notebook covers:**
1. Document size limits and fallback behaviour
2. PDF parsing: comparing two conversion engines
3. Chunking strategies: header-based, fixed-size, and paragraph-aware
4. Chunk size distributions and the embedding token limit
5. Tables, images, and Excel files
6. How chunking choices affect retrieval quality

**Prerequisite:** `conversational-toolkit` and `backend` must be installed in editable mode.


---

## Setup


In [ ]:
import re
import warnings

from markitdown import MarkItDown  # type: ignore[import-untyped]
from docling.document_converter import DocumentConverter  # type: ignore[import-untyped]

from conversational_toolkit.chunking.excel_chunker import ExcelChunker

from sme_kt_zh_collaboration_rag.feature0_baseline_rag import DATA_DIR, MAX_FILE_SIZE_MB
from sme_kt_zh_collaboration_rag.feature0_ingestion import (
    analyze_chunks,
    char_histogram,
    compare_strategies,
    fixed_size_chunks,
    header_based_chunks,
    paragraph_aware_chunks,
)

warnings.filterwarnings("ignore")

# Pick a representative PDF from the corpus for demonstrations
sample_pdf = str(DATA_DIR / "EPD_pallet_relicyc_logypal1.pdf")
print(f"Sample file: {sample_pdf}")
print(f"Data directory: {DATA_DIR}")
print(f"File size limit: {MAX_FILE_SIZE_MB} MB")

---

## 1. Document Size Limits

Not every PDF can be parsed safely. Very large files (image-heavy catalogues, scanned books) can exhaust memory or take minutes to convert. The ingestion pipeline checks file size before attempting to parse and skips files above the limit.


In [ ]:
# Show file sizes in the data directory and flag files above the limit
print(f"{'File':<55}  {'Size (MB)':>9}  {'Status'}")
print("-" * 80)
for f in sorted(DATA_DIR.iterdir()):
    if not f.is_file():
        continue
    size_mb = f.stat().st_size / (1024 * 1024)
    status = "SKIP (too large)" if size_mb > MAX_FILE_SIZE_MB else "ok"
    flag = "  <--" if size_mb > MAX_FILE_SIZE_MB else ""
    print(f"  {f.name:<53}  {size_mb:>9.1f}  {status}{flag}")

---

## 2. PDF Parser Comparison: MarkItDown vs Docling

Before chunking, a PDF must be converted to plain text. Three engines are available:

| Engine | Approach | Strengths | Weaknesses |
|--------|----------|-----------|------------|
| `markitdown` | Converts via an intermediate format | Handles more file types (DOCX, PPTX) | Slower, sometimes loses structure |
| `docling` | AI-assisted layout analysis with ML models | Best table extraction, handles complex layouts, reading order | Slow (first run downloads models), heavy dependency |

The choice of parser affects which headings are detected, whether tables survive as Markdown, how much text is dropped from figures or sidebars, and overall parsing speed.

**Docling note:** On first use, Docling downloads its ML models. Subsequent runs reuse the cached models and are faster.

In [ ]:
import time

# Parse the same file with all three engines and compare output length
t0 = time.time()
md_markitdown = str(MarkItDown().convert(sample_pdf).text_content)
t_markitdown = time.time() - t0
t0 = time.time()
md_docling = DocumentConverter().convert(sample_pdf).document.export_to_markdown()
t_docling = time.time() - t0

print(f"{'Engine':<15}  {'Chars':>8}  {'Lines':>7}  {'Time':>8}")
print("─" * 45)
print(
    f"{'markitdown':<15}  {len(md_markitdown):>8,}  {len(md_markitdown.splitlines()):>7,}  {t_markitdown:>7.1f}s"
)
print(
    f"{'docling':<15}  {len(md_docling):>8,}  {len(md_docling.splitlines()):>7,}  {t_docling:>7.1f}s"
)

In [ ]:
def extract_headings(md_text):
    return [
        line.strip() for line in md_text.splitlines() if re.match(r"^#{1,4} ", line)
    ]


headings_markitdown = extract_headings(md_markitdown)
headings_docling = extract_headings(md_docling)

for label, headings in [
    ("markitdown", headings_markitdown),
    ("docling", headings_docling),
]:
    print(f"Headings found by {label} ({len(headings)}):")
    for h in headings[:15]:
        print(f"  {h}")
    print()

In [ ]:
# Table extraction comparison: count Markdown pipe tables in each parser's output
def count_tables(md_text):
    """Count Markdown pipe-table blocks (heuristic: lines starting with |)."""
    in_table = False
    count = 0
    for line in md_text.splitlines():
        stripped = line.strip()
        if stripped.startswith("|"):
            if not in_table:
                in_table = True
                count += 1
        else:
            in_table = False
    return count


for label, md in [
    ("markitdown", md_markitdown),
    ("docling", md_docling),
]:
    n_tables = count_tables(md)
    print(f"{label:<15}: {n_tables} table(s) detected")

# Show a sample table from docling (if any) -> docling's table extraction is most reliable
table_lines = []
in_table = False
for line in md_docling.splitlines():
    if line.strip().startswith("|"):
        table_lines.append(line)
        in_table = True
    elif in_table:
        break

if table_lines:
    print(f"\nFirst table from docling ({len(table_lines)} rows):")
    for row in table_lines:
        print(f"  {row}")
else:
    print("\nNo pipe tables found in docling output for this file.")

---

## 3. Chunking Strategies

Three strategies are available. The right choice depends on the document structure:

| Strategy | How it splits | Best for | Risk |
|----------|--------------|----------|------|
| **Header-based** | One chunk per Markdown heading section | Well-structured PDFs with clear sections | Unstructured docs produce very large or very small chunks |
| **Fixed-size** | Every N characters, with overlap | Uniform chunk sizes, predictable embedding behaviour | May cut mid-sentence or mid-table |
| **Paragraph-aware** | Merges paragraphs until target size | Flowing text, narrative documents | Size varies more than fixed |


In [ ]:
# Header-based chunking (default)
chunks_header = header_based_chunks(sample_pdf)
stats_header = analyze_chunks(chunks_header, "header_based")
print("Header-based chunks:")
print(stats_header)
print(char_histogram(chunks_header))

In [ ]:
# Fixed-size chunking
chunks_fixed = fixed_size_chunks(sample_pdf, chunk_size=800, overlap=100)
stats_fixed = analyze_chunks(chunks_fixed, "fixed_size_800")
print("Fixed-size chunks (800 chars, 100 overlap):")
print(stats_fixed)
print(char_histogram(chunks_fixed))

In [ ]:
# Paragraph-aware chunking
chunks_para = paragraph_aware_chunks(sample_pdf, target_chars=500)
stats_para = analyze_chunks(chunks_para, "paragraph_500")
print("Paragraph-aware chunks (target 500 chars):")
print(stats_para)
print(char_histogram(chunks_para))

In [ ]:
# Side-by-side comparison table
results = compare_strategies(sample_pdf)

---

## 4. The Embedding Token Limit

The embedding model `all-MiniLM-L6-v2` silently truncates input at 256 tokens (~1000 characters). Content beyond that limit is lost before it ever reaches the vector store.

The `>256tok` column in the comparison table above shows how many chunks exceed this limit.

**Options if many chunks are truncated:**
- Switch to a model with a higher token limit (e.g. `text-embedding-3-small` supports 8,191 tokens)
- Reduce chunk size
- Use fixed-size chunking to cap the maximum size


In [ ]:
# Show the token distribution across chunking strategies
from sme_kt_zh_collaboration_rag.feature0_ingestion import estimate_tokens

for strategy_name, (chunks, stats) in results.items():
    token_counts = [estimate_tokens(c.content) for c in chunks]
    over = sum(1 for t in token_counts if t > 256)
    pct = 100 * over / len(chunks) if chunks else 0
    print(
        f"{strategy_name:<15} {over:>4} / {len(chunks)} chunks exceed 256 tokens  ({pct:.0f}%)"
    )

---

## 5. Tables and Spreadsheets

Tables in PDFs are rendered as Markdown pipe tables after parsing. Quality depends on the table complexity and the parser engine.

Excel files are handled by a separate chunker: one Markdown table per sheet.


In [ ]:
# Find chunks that contain Markdown tables in the header-based output
table_chunks = [c for c in chunks_header if c.content.count("|") >= 4]
print(f"Chunks containing tables: {len(table_chunks)}")
if table_chunks:
    print("\nFirst table chunk:")
    print(table_chunks[0].content)

This converts to:
| MODEL                                 | Logypal 1    | 
|---------------------------------------|--------------| 
| Dimensions [mm] (LxWxH)               | 1200x800x138 | 
| Weight [kg]                           | 4,5          | 
| Dynamic load [kg]                     | 800          | 
| Static load [kg]                      | 1600         | 
| Stackable                             | yes          | 
| Number of pallets for standard stack  | 62           | 
| Stack heigh for lorry [mm]            | 2610         | 
| Weight for stack loaded in lorry [kg] | 310          |


In [ ]:
# Excel files: one chunk per sheet
xlsx_file = str(DATA_DIR / "ART_product_overview.xlsx")
excel_chunks = ExcelChunker().make_chunks(xlsx_file)
print(f"Excel chunks: {len(excel_chunks)}")
for c in excel_chunks:
    print(f"  Sheet: {c.title}")
    print(c.content)

|  | Product Category | ID | Product Name | Supplier | EPD |
| --- | --- | --- | --- | --- | --- |
|  | Tape | 50-100 | Pressure-Sensitive Hot Melt Carton Sealing Tape | ipg | yes |
|  |  | 50-101 | Water-Activated Tape | ipg | yes |
|  |  | 50-102 | tesapack ECO & ULTRA STRONG ecoLogo | tesa | no |
|  | Pallets | 32-100 | Noé pallet | CPR System | yes |
|  |  | 32-101 | Wooden pallet | CPR System | no |
|  |  | 32-102 | Plastic pallet | CPR System | no |
|  |  | 32-103 | Logypal 1 | Relicyc | yes |
|  |  | 32-104 | LogyLight | Relicyc | no |
|  |  | 32-105 | Plastic pallet EP 08 ® | StabilPlastik | yes |
|  | Cardboard boxes | 11-100 | Cartonpallet CMP Roserio | redbox | yes |
|  |  | 11-101 | Corrugated cardboard packaging | Grupak | yes |

---

## 6. How Chunking Affects Retrieval

The same question can retrieve very different chunks depending on the chunking strategy. Header-based chunks tend to be self-contained, while fixed-size chunks may contain partial context from adjacent sections.

Try the queries below across different chunk sets to see which strategy surfaces the right content.


In [ ]:
# Concrete example: compare what each strategy retrieves for the same query
query_terms = ["GWP", "global warming", "carbon"]

for strategy_name, (chunks, _) in results.items():
    hits = [
        c for c in chunks if any(t.lower() in c.content.lower() for t in query_terms)
    ]
    print(
        f"\n{strategy_name}: {len(hits)} / {len(chunks)} chunks mention {query_terms}"
    )
    if hits:
        print(
            f"  First hit ({len(hits[0].content)} chars): {hits[0].content[:500].strip()!r}"
        )

---

## Summary

### PDF Parsers

| Engine | Speed | Table quality | Heading detection | Best for |
|--------|-------|---------------|-------------------|----------|
| `markitdown` | Medium | Moderate | Variable | Multi-format corpora (DOCX, PPTX, PDF) |
| `docling` | Slow (ML models) | Excellent | Good | Complex layouts, scanned PDFs, rich tables |

### Chunking Strategies

| Aspect | Header-based | Fixed-size | Paragraph-aware |
|--------|-------------|-----------|-----------------|
| Chunk boundaries | Section headings | Every N chars | Paragraph breaks |
| Size predictability | Low | High | Medium |
| Context preservation | High (one topic per chunk) | Medium (overlap helps) | Medium |
| Truncation risk | High for long sections | Low (tune chunk_size) | Medium |
| Best for | Structured reports, EPDs | Any document | Flowing narrative text |

**Key takeaways:**
- `docling` handles complex layouts and scanned PDFs it does not miss title pages.
- Header-based chunking works well for structured EPD and policy documents. For less structured documents, fixed-size chunking with a size below the 256-token limit is safer.